In [11]:
from pymongo import MongoClient

client = MongoClient("mongodb://localhost:27017")
#client.sample_mflix #온점 표기법 -> 온점 포함 // 숫자로 시작 하는 경우 사용X
db = client["sample_mflix"] #대괄호 표기법

users = db.users
comments = db.comments
movies = db.movies

print("ok", db.name, "collections :", [c for c in db.list_collection_names() if c in {"users", "comments", "movies"}])

print(db)

ok sample_mflix collections : ['users', 'movies', 'comments']
Database(MongoClient(host=['localhost:27017'], document_class=dict, tz_aware=False, connect=True), 'sample_mflix')


In [17]:
loyal_users_Pipeline = [
    {
        "$lookup": {
            "from": "comments",
            "localField": "email",
            "foreignField": "email",
            "as": "c"
        }
    },

    {
        "$addFields": {
            "commentsCount": {"$size": "$c"},
            "avgTextLen": {
                "$avg": {
                    "$map": {
                        "input": "$c",
                        "as": "x",
                        "in": {
                            "$strLenCP": {"$ifNull": ["$$x.text", ""]}
                        }
                    }
                }
            },
            "lastCommentDate": {"$max": "$c.date"}
        }
    },

    {"$match": {"commentsCount": {"$gte": 10}}},
    {"$sort": {"commentsCount": -1, "commentsCount": -1, "lastCommentDate": -1}},
    {
        "$project": {
            "_id": 0,
            "name": 1,
            "email": 1,
            "commentsCount": 1,
            "avgTextLen": 1,
            "lastCommentDate": 1
        }
    }
]

loyal_users = list(users.aggregate(loyal_users_Pipeline))

loyal_users[0:5]

# NoSQL -> 현업 실주 개발자들은 JOIN -> 서로 다른 데이터 테이블 혹은 컬렉션을 하나로 연결해서 가져와 처리하려면  해당 데이터의 양이 많아짐
# 방대해진 데이터를 처리하기 위한 하드웨어에 많은 무리가 가고 -> 하드웨어 성능 & 사양 제약

[{'name': 'Mace Tyrell',
  'email': 'roger_ashton-griffiths@gameofthron.es',
  'commentsCount': 331,
  'avgTextLen': 152.49244712990938,
  'lastCommentDate': datetime.datetime(2017, 8, 2, 1, 37, 26)},
 {'name': 'Missandei',
  'email': 'nathalie_emmanuel@gameofthron.es',
  'commentsCount': 327,
  'avgTextLen': 153.0948012232416,
  'lastCommentDate': datetime.datetime(2017, 9, 11, 16, 52, 51)},
 {'name': 'The High Sparrow',
  'email': 'jonathan_pryce@gameofthron.es',
  'commentsCount': 315,
  'avgTextLen': 152.68253968253967,
  'lastCommentDate': datetime.datetime(2017, 7, 10, 4, 58, 23)},
 {'name': 'Sansa Stark',
  'email': 'sophie_turner@gameofthron.es',
  'commentsCount': 308,
  'avgTextLen': 153.42532467532467,
  'lastCommentDate': datetime.datetime(2017, 5, 2, 16, 29, 50)},
 {'name': 'Rodrik Cassel',
  'email': 'ron_donachie@gameofthron.es',
  'commentsCount': 305,
  'avgTextLen': 151.75737704918032,
  'lastCommentDate': datetime.datetime(2017, 9, 2, 6, 1, 19)}]

In [18]:
movies_report_Pipeline = [
    {
        "$facet": {
            "$latest5": [
                {"$sort": {"year": -1}},
                {"$limit": 5},
                {"$project": {"_id": 0, "title": 1, "year": 1}}
            ],
            "$highRatedCount": [
                {"$match": {"imdb.rating": {"$gte": 8}}},
                {"$count": "count"}
            ],
            "genresTop10": [
                {"$unwind": "$genres"},
                {"$group": {"_id": "$genres", "count": {"$sum": 1}}},
                {"$sort": {"count": -1}},
                {"$limit": 10},
                {"$project": {"_id": 0, "genre": "$_id", "count": 1}}
            ],
            "yearlyAvgRecent10": [
                {"$group": {"_id": "$year", "avgRating": {"$avg": "$imdb.rating"}}},
                {"$sort": {"_id": -1}},
                {"$limit": 10},
                {"$project": {"_id": 0, "year": "_id", "avgRating": 1}}
            ]
        }
    }
]
movies.aggregate(movies_report_Pipeline)

OperationFailure: FieldPath field names may not start with '$'. Consider using $getField or $setField., full error: {'ok': 0.0, 'errmsg': "FieldPath field names may not start with '$'. Consider using $getField or $setField.", 'code': 16410, 'codeName': 'Location16410'}